In [ ]:
!pip install transformers datasets accelerate


In [ ]:
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm.auto import tqdm


In [ ]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

policy_model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
ref_model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
ref_model.eval()


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
imdb = load_dataset("imdb")

def make_prompt(example):
    text = example["text"][:200]
    return {"prompt": "Review: " + text + "\nSentiment:"}

prompts = imdb["train"].map(make_prompt)["prompt"][:100]  # 100 prompts for fast training
len(prompts)


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

100

In [ ]:
gen_pipe = pipeline("text-generation", model=policy_model, tokenizer=tokenizer, device=0)
sent_pipe = pipeline("sentiment-analysis", device=0)

def score(text):
    r = sent_pipe(text)[0]
    return r["score"] * (+1 if r["label"] == "POSITIVE" else -1)

pairs = []

for prompt in tqdm(prompts):
    y1 = gen_pipe(prompt, max_new_tokens=60, do_sample=True)[0]["generated_text"]
    y2 = gen_pipe(prompt, max_new_tokens=60, do_sample=True)[0]["generated_text"]

    s1, s2 = score(y1), score(y2)

    chosen = y1 if s1 >= s2 else y2
    rejected = y2 if s1 >= s2 else y1

    pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})

dataset = Dataset.from_list(pairs)
dataset = dataset.train_test_split(0.1)
train_dataset = dataset["train"]


Device set to use cuda:0
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


  0%|          | 0/100 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
def tokenize(example):
    return {
        "prompt_ids": tokenizer(example["prompt"], truncation=True, padding="max_length", max_length=128)["input_ids"],
        "chosen_ids": tokenizer(example["chosen"], truncation=True, padding="max_length", max_length=128)["input_ids"],
        "rejected_ids": tokenizer(example["rejected"], truncation=True, padding="max_length", max_length=128)["input_ids"],
    }

train_dataset = train_dataset.map(tokenize)


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

In [ ]:
from torch.utils.data import DataLoader

def collate(batch):
    return {
        "prompt_ids": torch.tensor([b["prompt_ids"] for b in batch]).cuda(),
        "chosen_ids": torch.tensor([b["chosen_ids"] for b in batch]).cuda(),
        "rejected_ids": torch.tensor([b["rejected_ids"] for b in batch]).cuda(),
    }

loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate)


In [ ]:
def seq_logprob(model, ids):
    attn = (ids != tokenizer.pad_token_id)
    out = model(ids, attention_mask=attn)
    logits = out.logits
    logprobs = torch.log_softmax(logits, dim=-1)
    token_lp = logprobs.gather(2, ids.unsqueeze(-1)).squeeze(-1)
    lp = (token_lp * attn).sum(-1)
    return lp

def dpo_loss(policy, ref, batch, beta=0.1):
    chosen = batch["chosen_ids"]
    rejected = batch["rejected_ids"]

    pi_c = seq_logprob(policy, chosen)
    pi_r = seq_logprob(policy, rejected)

    with torch.no_grad():
        ref_c = seq_logprob(ref, chosen)
        ref_r = seq_logprob(ref, rejected)

    loss = -torch.log(torch.sigmoid(beta * ((pi_c - pi_r) - (ref_c - ref_r)))).mean()
    return loss


In [ ]:
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6)

for epoch in range(1):
    for batch in loader:
        loss = dpo_loss(policy_model, ref_model, batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print("Epoch loss:", loss.item())


Epoch loss: 0.20463182032108307


In [ ]:
def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=60, do_sample=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompt = "Write a positive review for a sci-fi movie:"

print("BEFORE DPO:")
print(generate(ref_model, prompt))

print("\nAFTER DPO:")
print(generate(policy_model, prompt))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


BEFORE DPO:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Write a positive review for a sci-fi movie: You deserve it. The writer gives you a recommendation. The movie stars, and you're really lucky if I have your approval. I'm sure you're right. Don't have your reasons, and don't try to find them.

You've asked me about my favorite sci-fi movie

AFTER DPO:
Write a positive review for a sci-fi movie:

This is what a movie critic calls the "positive review":

After hearing it for the first time, what an unexpected experience it was, and what a lot of the reviewers have felt: So much so, that I can't even imagine what I want to be doing with my life
